## 流式输出

In [ ]:
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

#### values & updates

In [ ]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    initial_state: str
    node_a_output: str
    node_b_output: str

def a(state: State):
    return {
        "node_a_output": "节点A输出",
    }

def b(state: State):
    return {
        "node_b_output": "节点B输出",
    }


builder = StateGraph(State)
builder.add_node(a)
builder.add_node(b)
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("b", END)
graph = builder.compile()

for chunk in graph.stream(
        {"initial_state": "初始状态"},
        stream_mode=["values", "updates"]
):
    print(chunk)

#### messages

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, MessagesState


def llm_node(state: MessagesState):
    ai_msg = model.invoke(state["messages"])
    return {
        "messages": [ai_msg],
    }


builder = StateGraph(MessagesState)
builder.add_node(llm_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", END)
graph = builder.compile()

for message_chunk, metadata in graph.stream(
    {"messages": [HumanMessage("你好，请问如何才能学好大模型")]},
    stream_mode="messages",
):
    if message_chunk.content:
        print(message_chunk.content, end="", flush=True)
print()

#### checkpoints

In [ ]:
from langgraph.types import interrupt, Command
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    initial_state: str
    node_a_output: str
    node_b_output: str
    node_c_output: str

def a(state: State):
    return {
        "node_a_output": "节点A输出",
    }

def b(state: State):
    return {
        "node_b_output": "节点B输出",
    }

def c(state: State):
    interrupt("中断测试")
    return {
        "node_c_output": "节点C输出",
    }


builder = StateGraph(State)
builder.add_node(a)
builder.add_node(b)
builder.add_node(c)
builder.add_edge(START, "a")
builder.add_edge(START, "b")
builder.add_edge(["a", "b"], "c")
builder.add_edge("c", END)
graph = builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "7-1-1"}}
for chunk in graph.stream(
        {"initial_state": "初始状态"},
        config=config,
        stream_mode="checkpoints"
):
    print(chunk)

In [ ]:
for chunk in graph.stream(
        Command(resume=""),
        config=config,
        stream_mode="checkpoints"
):
    print(chunk)

#### tasks & debug

In [ ]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    initial_state: str
    node_a_output: str
    node_b_output: str

def a(state: State):
    return {
        "node_a_output": "节点A输出",
    }

def b(state: State):
    return {
        "node_b_output": "节点B输出",
    }


builder = StateGraph(State)
builder.add_node(a)
builder.add_node(b)
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("b", END)
graph = builder.compile()

for chunk in graph.stream(
        {"initial_state": "初始状态"},
        stream_mode=["tasks", "debug"]
):
    print(chunk)

#### custom

In [ ]:
from langgraph.runtime import Runtime
from typing import TypedDict

from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    initial_state: str
    node_a_output: str
    node_b_output: str

def a(state: State, runtime: Runtime):
    writer = runtime.stream_writer
    writer("节点A开始执行")
    return {
        "node_a_output": "节点A输出",
    }

def b(state: State, runtime: Runtime):
    writer = runtime.stream_writer
    writer("节点B开始执行")
    return {
        "node_b_output": "节点B输出",
    }


builder = StateGraph(State)
builder.add_node(a)
builder.add_node(b)
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("b", END)
graph = builder.compile()

for chunk in graph.stream(
        {"initial_state": "初始状态"},
        stream_mode=["values", "custom"]
):
    print(chunk)